In [1]:
from datetime import datetime, date, timedelta
from tqdm import tqdm
import pandas as pd
import math

In [3]:
path = "C:\\Users\\oscar\\big_ahh_files\\DW model\\game_fact.csv"
game_fact = pd.read_csv(path)
path = "C:\\Users\\oscar\\big_ahh_files\\DW model\\arena_change_fact.csv"
arena_fact = pd.read_csv(path)
path = "C:\\Users\\oscar\\big_ahh_files\\DW model\\arena_dim.csv"
arena_dim = pd.read_csv(path)
path = "C:\\Users\\oscar\\big_ahh_files\\DW model\\team_dim.csv"
team_dim = pd.read_csv(path)
gf = game_fact
arena_fact.columns

C:\Users\oscar\AppData\Local\Temp\ipykernel_10604\2388745545.py:2: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  game_fact = pd.read_csv(path)


Index(['Unnamed: 0', 'team_key', 'arena_key', 'arena_start_date'], dtype='object')

In [5]:
af = arena_fact
arena_dim.columns

Index(['arena_key', 'arena_name', 'arena_coor', 'arena_lat', 'arena_long',
       'nearest_airport', 'airport_coor', 'airport_lat', 'airport_long'],
      dtype='object')

In [7]:
ad = arena_dim[['arena_key', 'arena_lat', 'arena_long']]
td = team_dim

In [9]:
#script converting cartesian coordinates of each arena from dtm to decimal
import re

def dms2dd(degrees, minutes, seconds, direction):
    dd = float(degrees) + float(minutes)/60 + float(seconds)/(60*60);
    if direction == 'E' or direction == 'S':
        dd *= -1
    return dd;

def dtm_to_dec(dms):
    parts = re.split('[^\d\w]+', dms)
    coor = dms2dd(parts[0], parts[1], parts[2], parts[3])
 
    return (coor)

for col in ['arena_lat', 'arena_long']:
    ad[col] = ad[col].apply(dtm_to_dec)
ad.head()

<>:11: SyntaxWarning: invalid escape sequence '\d'
<>:11: SyntaxWarning: invalid escape sequence '\d'
C:\Users\oscar\AppData\Local\Temp\ipykernel_10604\3174639067.py:11: SyntaxWarning: invalid escape sequence '\d'
  parts = re.split('[^\d\w]+', dms)
C:\Users\oscar\AppData\Local\Temp\ipykernel_10604\3174639067.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ad[col] = ad[col].apply(dtm_to_dec)


,arena_key,arena_lat,arena_long
0,1,38.580278,121.499444
1,2,33.757222,84.396389
2,3,44.979444,93.276111
3,4,33.445833,112.071389
4,5,40.682500,73.975000


In [11]:
def string_to_date(date_str):
    return datetime.strptime(date_str, "%Y-%m-%d")

def date_to_string(date):
    return date.strftime('%Y-%m-%d')

#get the arena_key based on the game_date and location_team_key
def get_current_arena(game_date, team_key):
    arena_hist = af.loc[af['team_key'] == team_key]
    if arena_hist.shape[0] == 1:
        return arena_hist['arena_key'].values[0]
    arena_hist = arena_hist.sort_values(by='arena_start_date', ascending=False)

    for index, row in arena_hist.iterrows():
        if string_to_date(row['arena_start_date']) < string_to_date(game_date):
            return row['arena_key']
        else:
            continue

def concat_location_team_key(df):
    location_team_keys = []
    for index, row in df.iterrows():
        if row['home?'] == 'home':
            location_team_keys.append(row['team_key'])
        else:
            location_team_keys.append(row['opposing_team_key'])

    loc_keys_df = pd.DataFrame({'location_team_key':location_team_keys})
    df.reset_index(inplace=True)
    df = pd.concat([df, loc_keys_df], axis=1)
    return df

def concat_location_arena_key(df):
    game_dates = list(df['game_date'])
    location_team_keys = list(df['location_team_key'])
    location_arena_keys = [0, ] * len(location_team_keys)

    for i in range(len(location_team_keys)):
        location_arena_keys[i] = get_current_arena(game_dates[i], location_team_keys[i])

    arena_key_df = pd.DataFrame({'location_arena_key':location_arena_keys})
    df.reset_index(inplace=True)
    df = pd.concat([df, arena_key_df], axis=1)
    return df

def concat_arena_coordinates(df):
    df = df.rename(columns={'location_arena_key':'arena_key'})
    df = pd.merge(df, ad, on='arena_key', how='inner')
    df = df.rename(columns={'arena_key':'location_arena_key'})
    return df

def haversine_miles(lat1, lon1, lat2, lon2):
    """
    Calculate the Haversine distance between two points on the Earth in miles.
    
    Parameters:
        lat1, lon1: Latitude and longitude of the first point (in decimal degrees)
        lat2, lon2: Latitude and longitude of the second point (in decimal degrees)
        
    Returns:
        Distance between the two points in miles.
    """
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.asin(math.sqrt(a))
    
    # Radius of Earth in miles
    R = 3958.8
    
    distance = R * c
    return distance

def coor_from_arena_key(arena_key):
    return ad.loc[ad['arena_key'] == arena_key].values[0]

def sort_df_by_time(df):
    df = df.sort_values(by='game_date')
    df.drop('level_0', axis=1, inplace=True)
    df.reset_index(inplace=True)
    return df

def concat_distance_travel(df):
    start_team_key = df.loc[0, 'team_key']
    first_game_date = df.loc[0, 'game_date']
    current_arena_key = get_current_arena(first_game_date, start_team_key) #assume starting position from home arena key

    distance_traveled = []
    for index, row in df.iterrows():
        if row['location_team_key'] != current_arena_key:
            temp, last_lat, last_long = coor_from_arena_key(current_arena_key)
            temp, current_lat, current_long = coor_from_arena_key(row['location_team_key'])
            distance = haversine_miles(last_lat, last_long, current_lat, current_long)
            distance_traveled.append(distance)
        else:
            distance_traveled.append(0)

    dist_df = pd.DataFrame({'distance_traveled': distance_traveled})
    df.drop('level_0', axis=1, inplace=True)
    df.reset_index(inplace=True)
    df = pd.concat([df, dist_df], axis=1)
    return df

def modify_data():
    modified_seasons = []
    for player_key in tqdm(gf['player_key'].unique()):
        all_player_games = gf.loc[gf['player_key'] == player_key, :]
        for season in all_player_games['season'].unique():
            season_df = all_player_games.loc[all_player_games['season'] == season, :]
            season_df = concat_location_team_key(season_df)
            season_df = concat_location_arena_key(season_df)
            season_df = concat_arena_coordinates(season_df)
            season_df = sort_df_by_time(season_df)
            season_df = concat_distance_travel(season_df)
            modified_seasons.append(season_df)

    main_df = modified_seasons[0]
    for season in modified_seasons[1:]:
        main_df = pd.concat([main_df, season], axis=0)
    return main_df
    print("Original game_fact.csv shape: ", gf.shape)
    print("Modified game_fact.csv data shape: ", main_df.shape)

modified_game_fact = modify_data()

100%|██████████| 1199/1199 [05:04<00:00,  3.94it/s]


In [12]:
print("Original game_fact.csv shape: ", gf.shape)
print("Modified game_fact.csv data shape: ", modified_game_fact.shape)

Original game_fact.csv shape:  (262755, 35)
Modified game_fact.csv data shape:  (262472, 42)


In [15]:
mod = modified_game_fact
print(len(mod['player_key'].unique()))
print(len(gf['player_key'].unique()))

print(len(mod['game_id'].unique()))
print(len(gf['game_id'].unique()))

missed_games = [game for game in gf['game_id'].unique() if game not in mod['game_id'].unique()]
print(len(missed_games))

1199
1199
10219
10232
13


In [16]:
misfit_team_keys = []
total_records = []
for game_id in missed_games:
    for team_key in gf.loc[gf['game_id'] == game_id, 'team_key'].unique():
        if team_key not in misfit_team_keys:
            misfit_team_keys.append(team_key)
    for team_key in gf.loc[gf['game_id'] == game_id, 'opposing_team_key'].unique():
        if team_key not in misfit_team_keys:
            misfit_team_keys.append(team_key)

    total_records.append(len(list(gf.loc[gf['game_id'] == game_id, 'team_key'])))

print(misfit_team_keys) #all will be greater than three digits meaning they are an all-start team

[112151, 111353, 111354, 123554, 32, 31, 130580, 130579, 130754, 111386, 111387, 130581]


In [23]:
print(sum(total_records))
print(gf.shape[0] - mod.shape[0])

283
283


In [27]:
#these are just all-star games and am excluding them from the record because they are not regular season games.

In [29]:
path = 'C:\\Users\\oscar\\big_ahh_files\\DW model\\game_fact(2).csv'
mod.to_csv(path)